# Training

In [1]:
import json
import subprocess
from datetime import datetime
import h5py
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime
import pandas as pd
from torch.utils.data import Dataset
import yaml
from surya.utils.data import build_scalers
from surya.datasets.helio import transform
import numpy as np
import xarray as xr
import torch
from surya.models.helio_spectformer import HelioSpectFormer
from downstream_examples.solar_flare_forcasting.models import HelioSpectformer2D
from sklearn.model_selection import LeaveOneOut
from torch.utils.data import Subset,DataLoader
import numpy as np
from surya.models.helio_spectformer import HelioSpectFormer
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from peft import LoraConfig, get_peft_model

/home/amritm/projects/surya-tec-forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/amritm/projects/surya-tec-forecasting/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

SOLAR_DIR = RAW_DIR / "solar_observations"
TEC_HDF5_DIR = RAW_DIR / "tec_hdf5"
TEC_GRID_DIR = PROCESSED_DIR / "tec_grids"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
RESULTS_DIR = OUTPUT_DIR / "results"
FEATURES_DIR = OUTPUT_DIR / "features" 
PREDICTION_DIR = OUTPUT_DIR / "predictions"

COMPARISON_DIR = FIGURE_DIR / "predicted_tec_maps"


LATENT_DIR = FEATURES_DIR / "latent_features"
POOLED_DIR = FEATURES_DIR / "pooled_features"
SPATIAL_DIR = FEATURES_DIR / "spatial_features"

FLARE_MANIFEST = DATA_DIR / "flare_manifest.csv"
SURYA_DIR = PROJECT_ROOT.parent / "Surya"
SURYA_MODEL_DIR = SURYA_DIR / "data" / "Surya-1.0"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
with open(Path(RESULTS_DIR) / "query_result.json", "r") as f:
    data = json.load(f)


In [5]:
records = data['records']

m_flares = [
    flare for flare in records if flare["flare_class"].startswith('M')
]

x_flares = [
    flare for flare in records if flare["flare_class"].startswith('X')
]
selected_flares = m_flares[:15] + x_flares[:15]

## Dataset Loader

Load scalers from Surya (Normalize into shape $X\in{R^{13*2*4096*4096}}$ )

In [6]:
class FlareDataset(Dataset):
    def __init__(self,manifest_path):
        self.df = pd.read_csv(manifest_path)
        with open(Path(SURYA_MODEL_DIR) / "scalers.yaml") as f:
            scaler_info = yaml.safe_load(f)

        scalers = build_scalers(scaler_info)
        self.channels = [
            "aia94", "aia131", "aia171", "aia193",
            "aia211", "aia304", "aia335", "aia1600",
            "hmi_m", "hmi_bx", "hmi_by", "hmi_bz", "hmi_v"
        ]
        self.means = np.array([scalers[ch].mean for ch in self.channels])
        self.stds = np.array([scalers[ch].std for ch in self.channels])
        self.scale_factors = np.array([scalers[ch].sl_scale_factor for ch in self.channels])
        self.epsilons = np.array([scalers[ch].epsilon for ch in self.channels])


    def __len__(self):
        return len(self.df)
    def load_solar(self, path):
        path = PROJECT_ROOT / path
        with xr.open_dataset(
            path,
            engine="h5netcdf",
            chunks=None,
            cache=False
        ) as ds:
            solar = ds[self.channels].to_array().load().to_numpy()

        return transform(
            solar,
            self.means,
            self.stds,
            self.scale_factors,
            self.epsilons
        )
    def load_tec(self, path):  
        path = PROJECT_ROOT / path      
        with np.load(path) as data:
            tec = data["tec"]
            tec_mask = data["tec_mask"]

        return tec, tec_mask

    def __getitem__(self, idx):
        flare = self.df.iloc[idx]

        solar_t5 = self.load_solar(flare["solar_t-5"])
        solar_t4 = self.load_solar(flare["solar_t-4"])

        solar = np.stack([solar_t5, solar_t4], axis=1)
        tec, tec_mask = self.load_tec(flare["tec_grid_path"])

        return {
            "solar": torch.from_numpy(solar),
            "tec": torch.from_numpy(tec),
            "tec_mask": torch.from_numpy(tec_mask),
            "flare_class": flare["flare_class"],
            "peak_time": flare["peak_time"]
        }
    def get_flares_by_class(self, flare_class):
        return self.df[self.df["flare_class"] == flare_class]

In [7]:
dataset = FlareDataset(DATA_DIR/"flare_manifest.csv")


In [19]:
print(len(dataset))

sample = dataset[0]

print(sample["solar"].shape)
print(sample["solar"].dtype)
print(torch.isfinite(sample["solar"]).all())

30
torch.Size([13, 2, 4096, 4096])
torch.float32
tensor(True)


In [20]:
flare15 = dataset.__getitem__(idx=15)
flare15

{'solar': tensor([[[[-2.7955e-01, -4.5286e-01, -4.7089e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.4036e-01],
           [-3.6507e-01, -3.1025e-01, -4.9601e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.4036e-01],
           [-3.9150e-01, -2.9382e-01, -4.9627e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.4036e-01],
           ...,
           [-4.5121e-01, -4.4202e-01, -3.7811e-01,  ..., -2.0305e-01,
            -2.0305e-01, -5.4036e-01],
           [-3.8197e-01, -3.4819e-01, -3.7878e-01,  ..., -1.1571e-01,
            -1.7886e-01, -5.4036e-01],
           [-5.4036e-01, -1.4429e-01, -4.5878e-01,  ..., -1.1571e-01,
            -1.7886e-01, -5.4036e-01]],
 
          [[-4.4656e-01, -2.9547e-01, -2.9696e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.4036e-01],
           [-4.9893e-01, -5.4036e-01, -5.4036e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.4036e-01],
           [-4.9165e-01, -5.4036e-01, -4.2959e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.40

In [48]:
# Inspect Flares according to class
dataset.get_flares_by_class("M9.3")

,flare_class,start_time,peak_time,end_time,active_region_number,harp_number,solar_t-5,solar_t-4,tec_t-3,tec_t-2,...,tec_t+2,tec_t+3,tec_t-3_hdf5,tec_t-2_hdf5,tec_t-1_hdf5,tec_t+0_hdf5,tec_t+1_hdf5,tec_t+2_hdf5,tec_t+3_hdf5,tec_grid_path
1,M9.3,2011-07-30 02:04:00,2011-07-30 02:09:00,2011-07-30 02:12:00,11261,750,solar_observations/20110729_2100.nc,solar_observations/20110729_2200.nc,figures/tec_maps/tec_20110729_230730.png,figures/tec_maps/tec_20110730_000730.png,...,figures/tec_maps/tec_20110730_040730.png,figures/tec_maps/tec_20110730_050730.png,tec_hdf5/gps110729g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_grids/flare_001.npz
3,M9.3,2011-08-04 03:41:00,2011-08-04 03:57:00,2011-08-04 04:04:00,11261,750,solar_observations/20110803_2248.nc,solar_observations/20110803_2348.nc,figures/tec_maps/tec_20110804_005730.png,figures/tec_maps/tec_20110804_015730.png,...,figures/tec_maps/tec_20110804_055730.png,figures/tec_maps/tec_20110804_065730.png,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_grids/flare_003.npz


$TEC_{i,j} = \frac{1}{N_{i,j}}\sum_{k\in{B_{i,j}}}{TEC_k}$

For this experiment, we will be using 1.0° resolution.

### 1. Loading LoRA Trained Surya

In [7]:
with open (Path(SURYA_DIR) / "downstream_examples/solar_flare_forcasting/config.yaml","r") as f:
    config = yaml.safe_load(f)


In [23]:
config["model"]["ft_unembedding_type"] = "linear"
config["model"]["ft_out_chans"] = 7

Instead of using Surya's in-built Linear Decoder, our TEC Prediction head will use a custom configuration.

In [52]:
# print(config["model"]["model_type"])
# print(config["model"]["embed_dim"])
# print(config["model"]["time_embedding"])

In [24]:
import torch

model = HelioSpectformer2D(
    img_size=config["model"]["img_size"],
    patch_size=config["model"]["patch_size"],
    in_chans=config["model"]["in_channels"],
    embed_dim=config["model"]["embed_dim"],
    time_embedding=config["model"]["time_embedding"],
    depth=config["model"]["depth"],
    n_spectral_blocks=config["model"]["spectral_blocks"],
    num_heads=config["model"]["num_heads"],
    mlp_ratio=config["model"]["mlp_ratio"],
    drop_rate=config["model"]["drop_rate"],
    window_size=config["model"]["window_size"],
    dp_rank=config["model"]["dp_rank"],
    learned_flow=config["model"]["learned_flow"],
    use_latitude_in_learned_flow=config["use_latitude_in_learned_flow"],
    init_weights=config["model"]["init_weights"],
    checkpoint_layers=config["model"]["checkpoint_layers"],
    rpe=config["model"]["rpe"],
    finetune=True,
    dtype=torch.float32,
    config=config,
)

In [25]:
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

Parameters: 364.22M


In [26]:
checkpoint = torch.load(
    Path(SURYA_DIR) / "data/Surya-1.0/surya.366m.v1.pt",
    map_location="cpu",
    weights_only=True
)

model_state = model.state_dict()

filtered_checkpoint = {
            k: v
            for k, v in checkpoint.items()
            if k in model_state and v.shape == model_state[k].shape
}
print(f"Matched: {len(filtered_checkpoint)} / {len(checkpoint)} tensors")

Matched: 157 / 159 tensors


In [27]:
for k, v in checkpoint.items():
    if k not in filtered_checkpoint:
        print(k, v.shape)

unembed.unembed.0.weight torch.Size([3328, 1280, 1, 1])
unembed.unembed.0.bias torch.Size([3328])


In [28]:
# model_state.update(filtered_checkpoint)
model.load_state_dict(filtered_checkpoint, strict=False)

print(torch.equal(
    model.state_dict()["embedding.patch_embed.proj.weight"],
    checkpoint["embedding.patch_embed.proj.weight"]
))

True


In [29]:
for name,module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        if "attn" in name:
            print(name, module.weight.shape)

backbone.blocks_attention.0.attn.qkv torch.Size([3840, 1280])
backbone.blocks_attention.0.attn.proj torch.Size([1280, 1280])
backbone.blocks_attention.0.attn.to_dynamic_projection torch.Size([64, 1280])
backbone.blocks_attention.1.attn.qkv torch.Size([3840, 1280])
backbone.blocks_attention.1.attn.proj torch.Size([1280, 1280])
backbone.blocks_attention.1.attn.to_dynamic_projection torch.Size([64, 1280])
backbone.blocks_attention.2.attn.qkv torch.Size([3840, 1280])
backbone.blocks_attention.2.attn.proj torch.Size([1280, 1280])
backbone.blocks_attention.2.attn.to_dynamic_projection torch.Size([64, 1280])
backbone.blocks_attention.3.attn.qkv torch.Size([3840, 1280])
backbone.blocks_attention.3.attn.proj torch.Size([1280, 1280])
backbone.blocks_attention.3.attn.to_dynamic_projection torch.Size([64, 1280])
backbone.blocks_attention.4.attn.qkv torch.Size([3840, 1280])
backbone.blocks_attention.4.attn.proj torch.Size([1280, 1280])
backbone.blocks_attention.4.attn.to_dynamic_projection torch.Si

In [34]:
targets = ["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]

for target in targets:
    matches = [
        name for name, module in model.named_modules()
        if name.endswith(target) and isinstance(module, torch.nn.Linear)
    ]
    print(f"{target}: {len(matches)} matches")

q_proj: 0 matches
v_proj: 0 matches
k_proj: 0 matches
out_proj: 0 matches
fc1: 10 matches
fc2: 10 matches


```
# Note: We will first test Surya LoRA Config using two out of three linear attention projections (query,key,value and output projections) before testing dynamic projection. 
```

In [36]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["attn.qkv", "attn.proj"],
    lora_dropout=.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model = model.to(device)

/home/amritm/projects/surya-tec-forecasting/.venv/lib/python3.12/site-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/amritm/projects/surya-tec-forecasting/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [63]:
model.print_trainable_parameters()

trainable params: 491,520 || all params: 364,713,984 || trainable%: 0.1348


In [37]:
for name,param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

base_model.model.base_model.model.backbone.blocks_attention.0.attn.qkv.lora_A.default.weight torch.Size([8, 1280])
base_model.model.base_model.model.backbone.blocks_attention.0.attn.qkv.lora_B.default.weight torch.Size([3840, 8])
base_model.model.base_model.model.backbone.blocks_attention.0.attn.proj.lora_A.default.weight torch.Size([8, 1280])
base_model.model.base_model.model.backbone.blocks_attention.0.attn.proj.lora_B.default.weight torch.Size([1280, 8])
base_model.model.base_model.model.backbone.blocks_attention.1.attn.qkv.lora_A.default.weight torch.Size([8, 1280])
base_model.model.base_model.model.backbone.blocks_attention.1.attn.qkv.lora_B.default.weight torch.Size([3840, 8])
base_model.model.base_model.model.backbone.blocks_attention.1.attn.proj.lora_A.default.weight torch.Size([8, 1280])
base_model.model.base_model.model.backbone.blocks_attention.1.attn.proj.lora_B.default.weight torch.Size([1280, 8])
base_model.model.base_model.model.backbone.blocks_attention.2.attn.qkv.lora_

In [38]:
print("Patch size:", model.patch_size)
print("Embedding dimension:", model.embed_dim)
print("Global tokens:", model.nglo)
print("Fine-tuning mode:", model.finetune)

Patch size: 16
Embedding dimension: 1280
Global tokens: 0
Fine-tuning mode: True


### 2. Training

Input: $ X \in \mathbb{R}^{13*2*4096*4096}$

Predictions: $\hat{Y} \in \mathbb{R}^{7*180*360}$

$L = (\frac{4096}{16})^2 = 256^2 = 65,356$
, $D = 1280$

$Z_{latent representations} \in{\mathbb{R}^{B*L*1280}}$ = ~84M latent values per observation

$\mathcal{L}_{masked MSE} = \frac{\sum_{t,i,j}{M_{t,i,j}(\hat{Y}_{t,i,j}-Y_{t,i,j})^2}}{\sum_{t,i,j}{M_{t,i,j}}}$

```text
Potential Structure:
Surya transformer
       |
       v
Spatial tokens (B, L, D)
       |
       v
Reshape to (B, D, H', W')
       |
       v
1×1 Conv (reduce feature channels)
       |
       v
Spatial resizing
       |
       v
3×3 Conv + GELU
       |
       v
3×3 Conv (7 output channels)
       |
       v
TEC predictions (B, 7, 180, 360)
```

$X_{solar} = [X_{t-5}, X_{t-4}]$, where each observation has 13 channels.



In [8]:
flare_ids = np.arange(len(dataset))
loo = LeaveOneOut()

for fold, (train_idx, test_idx) in enumerate(loo.split(flare_ids)):
    train_dataset = Subset(dataset, train_idx.tolist())
    test_dataset = Subset(dataset, test_idx.tolist())

    trainloader = DataLoader(
        dataset=train_dataset,
        batch_size=1,
        shuffle=True
    )
    testloader = DataLoader(
        dataset=test_dataset,
        batch_size=1,
        shuffle=False
    )

    # print(f"Fold {fold+1}")
    # print(f"Training flares: {len(train_idx)}")
    # print(f"Testing flare: {test_idx[0]}")

    # held_out = dataset.df.iloc[test_idx[0]]

    # print(f"Held-out flare: {held_out['peak_time']}")
    # print(f"Flare class: {held_out['flare_class']}")

In [9]:
def masked_mse(pred,target,mask):
    squared_error = (pred-target).square()
    return (mask*squared_error).sum()/mask.sum().clamp(min=1)

Shape: $ [B, C, T, H, W] $ where:


B - Batch,
C - Channel,
T- Time,
H - Height,
W - Width

Latent Representation Shape: $[1, 65536, 1280]$

```text
Solar: torch.Size([1, 13, 2, 4096, 4096])
TEC: torch.Size([1, 7, 180, 360])
```

In [ ]:
""" 
We extract and cache Surya's latent representations for each flare to avoid redundant forward passes 
during subsequent experiments.
"""
# output_dir = Path("latent_features")
# output_dir.mkdir(exist_ok=True)

# model.eval()

# latent_features = []

# with torch.inference_mode():
#     for i in range(len(dataset)):
#         flare = dataset[i]

#         batch = {
#             "ts": flare["solar"].unsqueeze(0).to(device),
#             "time_delta_input" : torch.tensor(
#                 [[-1.0, 0.0]],
#                 device=device
#             )
#         }
#         tokens = HelioSpectFormer.forward(model, batch=batch)

#         torch.save(tokens.cpu(), output_dir / f"flare_{i:02d}.pt")

#         print(f"Saved flare {i + 1}/{len(dataset)}", flush=True)

In [73]:
# pooled_dir = Path("pooled_features")
# pooled_dir.mkdir(exist_ok=True)

# for i in range(len(dataset)):
#     tokens = torch.load(
#         Path("latent_features") / f"flare_{i:02d}.pt",
#         map_location="cpu",
#         weights_only=True
#     )

#     pooled = tokens.mean(dim=1)

#     torch.save(pooled, pooled_dir / f"flare_{i:02d}.pt")

#     print(f"Pooled flare {i + 1}/{len(dataset)}")

In [10]:
class TECLinearHead(nn.Module):
    def __init__(self, 
                input_dim=1280, 
                output_dim=7*18*36,  
                low_res=(36, 72),
                output_res=(180, 360),
                num_frames=7):
        super().__init__()
        self.low_res = low_res
        self.output_res = output_res
        self.num_frames = num_frames

        h,w = low_res

        self.linear = nn.Linear(input_dim, num_frames * h * w)

    def forward(self,x):
        x = self.linear(x)

        # Reshape X to 7 low-resolution TEC Maps
        x = x.reshape(-1,self.num_frames,
                      *self.low_res)

        #Upscale into the original TEC resolution
        if self.low_res != self.output_res:
            x = F.interpolate(
                x,
                size=self.output_res,
                mode="bilinear",
                align_corners=False
            )

        return x

In [75]:
# head = TECLinearHead()

# pooled = torch.load(
#     pooled_dir / "flare_00.pt",
#     map_location="cpu",
#     weights_only=True
# )

# prediction = head(pooled)

# print(prediction.shape)

In [76]:
# train_idx, test_idx = next(loo.split(range(len(dataset))))

# head = TECLinearHead().to(device)

# optimizer = optim.Adam(
#     head.parameters(),
#     lr=1e-3
# )



In [11]:
# COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
def plot_comparison_map(predicted_map, actual_map, flare_id, offset,experiment_name):
    """
    Compare predicted and actual TEC maps for a held-out flare.

    Both inputs should be 2D NumPy arrays with shape (H, W).
    """

    predicted_map = np.asarray(predicted_map)
    actual_map = np.asarray(actual_map)

    assert predicted_map.shape == actual_map.shape

    H,W = actual_map.shape

    lats = np.linspace(-90,90,H)
    lons = np.linspace(-180,180,W)

    # Shared color scale for meaningful comparison
    vmin = min(np.nanmin(predicted_map), np.nanmin(actual_map))
    vmax = max(np.nanmax(predicted_map), np.nanmax(actual_map))

    fig,axes = plt.subplots(
        1,2,
        figsize=(16,5),
        constrained_layout=True
    )
    for ax,data,title in zip(axes,[actual_map,predicted_map],["Actual TEC", "Predicted TEC"]):
        mesh = ax.pcolormesh(
            lons,
            lats,
            data,
            shading="auto",
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
        )

        ax.set_title(title)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xlim(-180,180)
        ax.set_ylim(-90,90)
    fig.colorbar(
        mesh,
        ax=axes,
        label="TEC",
        shrink=0.85
    )
    fig.suptitle(
        f"Flare {flare_id:03d} — TEC Forecast t{offset:+d}h",
        fontsize=14
    )
    output_dir = COMPARISON_DIR / experiment_name / f"flare_{flare_id:03d}"
    output_dir.mkdir(parents=True, exist_ok=True)
    save_path = output_dir / f"comparison_t{offset:+d}.png"
    if save_path.exists():
        return save_path
    
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return save_path

In [44]:
experiments = {
    "gap_18x36_20ep": {
        "feature_dir": POOLED_DIR,
        "input_dim": 1280,
        "low_res": (18, 36),
        "epochs": 20,
        "lr": 1e-3,
    },
    "spatial_4x4_20ep": {
        "feature_dir": SPATIAL_DIR,
        "input_dim": 20480,
        "low_res": (18, 36),
        "epochs": 20,
        "lr": 1e-4,
    },
    "gap_18x36_40ep": {
        "feature_dir": POOLED_DIR,
        "input_dim": 1280,
        "low_res": (18, 36),
        "epochs": 40,
        "lr": 1e-4,
    },
    "spatial_4x4_40ep": {
        "feature_dir": SPATIAL_DIR,
        "input_dim": 20480,
        "low_res": (18, 36),
        "epochs": 40,
        "lr": 1e-4,
    },
}

In [ ]:
def run_loocv(config, dataset, device,experiment_name):
    loo = LeaveOneOut()
    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(
        loo.split(range(len(dataset))), start=1
    ):
        torch.manual_seed(42 + fold)
        head = TECLinearHead(
            input_dim=config["input_dim"],
            low_res=config["low_res"]
        ).to(device)

        optimizer = optim.Adam(
            head.parameters(),
            lr=config["lr"]
        )

        for epoch in range(config["epochs"]):
            head.train()

            for i in train_idx:
                i = int(i)

                tokens = torch.load(
                    Path(config["feature_dir"]) / f"flare_{i:02d}.pt",
                    map_location=device,
                    weights_only=True
                )

                tec_path = dataset.df.iloc[i]["tec_grid_path"]
                tec, tec_mask = dataset.load_tec(tec_path)

                target = torch.from_numpy(tec).unsqueeze(0).to(device)
                mask = torch.from_numpy(tec_mask).unsqueeze(0).to(device)

                optimizer.zero_grad()

                prediction = head(tokens)
                loss = masked_mse(prediction, target, mask)

                loss.backward()
                optimizer.step()

        head.eval()

        with torch.no_grad():
            i = int(test_idx[0])

            tokens = torch.load(
                Path(config["feature_dir"]) / f"flare_{i:02d}.pt",
                map_location=device,
                weights_only=True
            )

            tec_path = dataset.df.iloc[i]["tec_grid_path"]
            tec, tec_mask = dataset.load_tec(tec_path)

            target = torch.from_numpy(tec).unsqueeze(0).to(device)
            mask = torch.from_numpy(tec_mask).unsqueeze(0).to(device)

            prediction = head(tokens)
            test_loss = masked_mse(prediction, target, mask)

            predicted_maps = prediction.squeeze(0).cpu().numpy()
            actual_maps = target.squeeze(0).cpu().numpy()
            valid_mask = mask.squeeze(0).cpu().numpy().astype(bool)

        output_dir = (
            PREDICTION_DIR
            / experiment_name
            / f"flare_{i:03d}"
        )

        output_dir.mkdir(parents=True, exist_ok=True)

        np.savez_compressed(
            output_dir / "predictions.npz",
            predicted=predicted_maps,
            actual=actual_maps,
            mask=valid_mask
        )

        fold_results.append(test_loss.item())

        print(
            f"Fold {fold:02d}/{len(dataset)} | "
            f"Held-out MSE: {test_loss.item():.4f}"
        )

    return np.array(fold_results)

In [14]:
def run_persistence(dataset):
    results = []

    for i in range(len(dataset)):
        tec_path = dataset.df.iloc[i]["tec_grid_path"]
        tec,tec_mask = dataset.load_tec(tec_path)

        # Use TEC at t=0 to predict t+1, t+2, t+3
        predicted = np.repeat(tec[3:4],3,axis=0)
        actual = tec[4:7]

        # Ensure that pixels used for predicted and actual are valid
        mask = tec_mask[4:7].astype(bool) & tec_mask[3:4].astype(bool)

        mse = np.mean((predicted[mask] - actual[mask]) ** 2) if mask.any() else np.nan

        output_dir = PREDICTION_DIR / "persistence" / f"flare_{i:03d}"
        output_dir.mkdir(parents=True, exist_ok=True)

        np.savez_compressed(
            output_dir / "predictions.npz",
            predicted=predicted,
            actual=actual,
            mask=mask
        )
        results.append(mse)
        print(f"Flare {i:20d} | Persistence MSE: {mse:.4f}")

    return np.array(results)

In [81]:
results = {}

for name, config in experiments.items():
    print(f"\nRunning experiment: {name}")

    fold_mse = run_loocv(config, dataset, device,name)

    results[name] = {
        "config": config,
        "fold_mse": fold_mse.tolist(),
        "mean": float(np.mean(fold_mse)),
        "median": float(np.median(fold_mse)),
        "std": float(np.std(fold_mse)),
        "min": float(np.min(fold_mse)),
        "max": float(np.max(fold_mse)),
    }


Running experiment: gap_18x36_20ep
Fold 01/30 | Held-out MSE: 38.0128
Fold 02/30 | Held-out MSE: 33.3309
Fold 03/30 | Held-out MSE: 49.7902
Fold 04/30 | Held-out MSE: 45.7549
Fold 05/30 | Held-out MSE: 41.1708
Fold 06/30 | Held-out MSE: 48.8436
Fold 07/30 | Held-out MSE: 53.0300
Fold 08/30 | Held-out MSE: 43.7290
Fold 09/30 | Held-out MSE: 57.3315
Fold 10/30 | Held-out MSE: 129.4893
Fold 11/30 | Held-out MSE: 37.9868
Fold 12/30 | Held-out MSE: 81.2315
Fold 13/30 | Held-out MSE: 99.1465
Fold 14/30 | Held-out MSE: 79.8506
Fold 15/30 | Held-out MSE: 200.9343
Fold 16/30 | Held-out MSE: 39.2898
Fold 17/30 | Held-out MSE: 155.1948
Fold 18/30 | Held-out MSE: 51.8677
Fold 19/30 | Held-out MSE: 52.0879
Fold 20/30 | Held-out MSE: 68.4320
Fold 21/30 | Held-out MSE: 51.9475
Fold 22/30 | Held-out MSE: 48.5748
Fold 23/30 | Held-out MSE: 108.2837
Fold 24/30 | Held-out MSE: 129.0858
Fold 25/30 | Held-out MSE: 118.4789
Fold 26/30 | Held-out MSE: 399.0089
Fold 27/30 | Held-out MSE: 182.5415
Fold 28/30 

In [15]:
persistence_results = run_persistence(dataset)
print(f"Mean persistence MSE: {np.nanmean(persistence_results):.4f}")

Flare                    0 | Persistence MSE: 21.7052
Flare                    1 | Persistence MSE: 15.4080
Flare                    2 | Persistence MSE: 30.5496
Flare                    3 | Persistence MSE: 13.4380
Flare                    4 | Persistence MSE: 23.9385
Flare                    5 | Persistence MSE: 18.9726
Flare                    6 | Persistence MSE: 43.3545
Flare                    7 | Persistence MSE: 42.3875
Flare                    8 | Persistence MSE: 80.2059
Flare                    9 | Persistence MSE: 47.7142
Flare                   10 | Persistence MSE: 23.7456
Flare                   11 | Persistence MSE: 33.7739
Flare                   12 | Persistence MSE: 32.6514
Flare                   13 | Persistence MSE: 48.1580
Flare                   14 | Persistence MSE: 58.4925
Flare                   15 | Persistence MSE: 23.7647
Flare                   16 | Persistence MSE: 223.2669
Flare                   17 | Persistence MSE: 23.3824
Flare                   18 

In [82]:
summary = pd.DataFrame({
    name: {
        "Mean MSE": result["mean"],
        "Median MSE": result["median"],
        "Std MSE": result["std"],
        "Min MSE": result["min"],
        "Max MSE": result["max"],
    }
    for name, result in results.items()
}).T

display(summary.sort_values("Mean MSE"))

,Mean MSE,Median MSE,Std MSE,Min MSE,Max MSE
spatial_4x4_40ep,87.935655,56.315355,75.661854,32.665585,377.532959
spatial_4x4_20ep,93.386705,57.300816,81.232279,35.036640,404.597107
gap_18x36_20ep,110.025302,62.881750,105.230056,33.330902,470.792938
gap_18x36_40ep,239.017794,119.655525,241.595928,22.978977,959.790649


In [ ]:
import json

with open(RESULTS_DIR / "experiment_2_results.json", "w") as f:
    json.dump(results, f, indent=2)

NameError: name 'results' is not defined

In [ ]:
for experiment_name in experiments:

    experiment_dir = PREDICTION_DIR / experiment_name

    for flare_dir in sorted(experiment_dir.glob("flare_*")):

        flare_id = int(flare_dir.name.split("_")[1])

        with np.load(flare_dir / "predictions.npz") as data:
            predicted_maps = data["predicted"]
            actual_maps = data["actual"]
            valid_mask = data["mask"].astype(bool)

        for j, offset in enumerate(range(-3, 4)):

            save_path = (
                COMPARISON_DIR
                / experiment_name
                / f"flare_{flare_id:03d}"
                / f"comparison_t{offset:+d}.png"
            )

            if save_path.exists():
                continue
            predicted_map = np.where(
                valid_mask[j],
                predicted_maps[j],
                np.nan
            )

            actual_map = np.where(
                valid_mask[j],
                actual_maps[j],
                np.nan
            )

            plot_comparison_map(
                predicted_map=predicted_map,
                actual_map=actual_map,
                flare_id=flare_id,
                offset=offset,
                experiment_name=experiment_name
            )

    print(f"Finished plotting: {experiment_name}")

Finished plotting: gap_18x36_20ep
Finished plotting: spatial_4x4_20ep
Finished plotting: gap_18x36_40ep
Finished plotting: spatial_4x4_40ep


In [16]:
experiment_name = "persistence"
experiment_dir = PREDICTION_DIR / experiment_name

for flare_dir in sorted(experiment_dir.glob("flare_*")):

    flare_id = int(flare_dir.name.split("_")[1])

    with np.load(flare_dir / "predictions.npz") as data:
        predicted_maps = data["predicted"]
        actual_maps = data["actual"]
        valid_mask = data["mask"].astype(bool)

    for j, offset in enumerate(range(1, 4)):

        save_path = (
            COMPARISON_DIR
            / experiment_name
            / f"flare_{flare_id:03d}"
            / f"comparison_t{offset:+d}.png"
        )

        if save_path.exists():
            continue

        predicted_map = np.where(
            valid_mask[j],
            predicted_maps[j],
            np.nan
        )

        actual_map = np.where(
            valid_mask[j],
            actual_maps[j],
            np.nan
        )

        plot_comparison_map(
            predicted_map=predicted_map,
            actual_map=actual_map,
            flare_id=flare_id,
            offset=offset,
            experiment_name=experiment_name
        )

print(f"Finished plotting: {experiment_name}")

Finished plotting: persistence


In [17]:
def plot_three_way_comparison(actual_map, persistence_map, surya_map, flare_id, offset, experiment_name):
    """
    Compare predicted and actual TEC maps for a held-out flare.

    Both inputs should be 2D NumPy arrays with shape (H, W).
    """

    surya_map = np.asarray(surya_map)
    actual_map = np.asarray(actual_map)
    persistence_map = np.asarray(persistence_map)

    assert surya_map.shape == actual_map.shape == persistence_map.shape

    H,W = actual_map.shape

    lats = np.linspace(-90,90,H)
    lons = np.linspace(-180,180,W)

    output_dir = COMPARISON_DIR / experiment_name / f"flare_{flare_id:03d}"
    output_dir.mkdir(parents=True, exist_ok=True)
    save_path = output_dir / f"comparison_t{offset:+d}.png"
    if save_path.exists():
        return save_path

    # Shared color scale for meaningful comparison
    vmin = min(np.nanmin(surya_map), np.nanmin(actual_map), np.nanmin(persistence_map))
    vmax = max(np.nanmax(surya_map), np.nanmax(actual_map), np.nanmax(persistence_map))


    fig,axes = plt.subplots(
        1,3,
        figsize=(21,5),
        constrained_layout=True
    )
    maps = [actual_map, persistence_map, surya_map]
    titles = ["Actual TEC", "Persistence TEC", "Surya Predicted TEC"]

    for ax,data,title in zip(axes,maps,titles):
        mesh = ax.pcolormesh(
            lons,
            lats,
            data,
            shading="auto",
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
        )

        ax.set_title(title)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xlim(-180,180)
        ax.set_ylim(-90,90)
    fig.colorbar(
        mesh,
        ax=axes,
        label="TEC",
        shrink=0.85
    )
    fig.suptitle(
        f"Flare {flare_id:03d} — TEC Forecast t{offset:+d}h",
        fontsize=14
    )
    
    
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return save_path


Three Way comparison between actual vs persistence vs spatial 4x4 (40 epochs)

In [18]:
experiment_name = "spatial_4x4_40ep"
comparison_name = f"{experiment_name}_vs_persistence"
experiment_dir = PREDICTION_DIR / experiment_name

for flare_dir in sorted(experiment_dir.glob("flare_*")):

    flare_id = int(flare_dir.name.split("_")[1])

    persistence_path = (
        PREDICTION_DIR
        / "persistence"
        / f"flare_{flare_id:03d}"
        / "predictions.npz"
    )

    with np.load(flare_dir / "predictions.npz") as data:
        predicted_maps = data["predicted"]
        actual_maps = data["actual"]
        valid_mask = data["mask"].astype(bool)

    with np.load(persistence_path) as data:
        persistence_maps = data["predicted"]
        persistence_mask = data["mask"].astype(bool)

    # Persistence only predicts +1h, +2h, +3h
    for j, offset in enumerate(range(1, 4)):

        save_path = (
            COMPARISON_DIR
            / comparison_name
            / f"flare_{flare_id:03d}"
            / f"comparison_t{offset:+d}.png"
        )

        if save_path.exists():
            continue

        surya_idx = offset + 3

        # Use pixels valid for both experiments
        mask = valid_mask[surya_idx] & persistence_mask[j]

        surya_map = np.where(
            mask, predicted_maps[surya_idx], np.nan
        )

        actual_map = np.where(
            mask, actual_maps[surya_idx], np.nan
        )

        persistence_map = np.where(
            mask, persistence_maps[j], np.nan
        )

        plot_three_way_comparison(
            actual_map=actual_map,
            persistence_map=persistence_map,
            surya_map=surya_map,
            flare_id=flare_id,
            offset=offset,
            experiment_name=comparison_name
        )

print(f"Finished plotting: {comparison_name}")


Finished plotting: spatial_4x4_40ep_vs_persistence


In [19]:
from zipfile import ZipFile, ZIP_DEFLATED

EXPERIMENT = "spatial_4x4_40ep_vs_persistence"

folds = [10, 21, 30]
offsets = [1, 3]  # Persistence only covers +1h to +3h

zip_path = PROJECT_ROOT / "surya_tec_three_way_comparison.zip"

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zf:
    for fold in folds:
        flare_id = fold - 1

        for offset in offsets:
            filename = f"comparison_t{offset:+d}.png"

            source = (
                COMPARISON_DIR
                / EXPERIMENT
                / f"flare_{flare_id:03d}"
                / filename
            )

            if not source.exists():
                print(f"Missing: {source}")
                continue

            archive_name = (
                f"Fold_{fold:02d}/"
                f"{filename}"
            )

            zf.write(source, arcname=archive_name)
            print(f"Added: {archive_name}")

print(f"\nSaved: {zip_path}")


Added: Fold_10/comparison_t+1.png
Added: Fold_10/comparison_t+3.png
Added: Fold_21/comparison_t+1.png
Added: Fold_21/comparison_t+3.png
Added: Fold_30/comparison_t+1.png
Added: Fold_30/comparison_t+3.png

Saved: /home/amritm/projects/surya-tec-forecasting/surya_tec_three_way_comparison.zip
